In [3]:
# =============================================================================
# NOTEBOOK: 01_agent1_task_extraction.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# AGENT 1 of 3 — Task Extraction & Automatability Classification
#   Input : raw operational interview transcript (any domain)
#   Method: Chain-of-Thought (Wei et al., 2022) — reason step-by-step, then emit
#           a structured list of work units, each with an AI-automatability grade
#           (full / partial / manual) AND an explicit rationale.
#   Output: artifacts/inference/agent1_work_units.json  (list[WorkUnit])
#
# Design principles realized here:
#   DP1 (role separation): this agent ONLY extracts + classifies. It never
#        estimates time (Agent 2) or computes ROI (Agent 3).
#   DP3 (mark the unknown): any field not stated in the interview is set to the
#        MISSING sentinel rather than guessed.
#   Transparency axis: every automatability grade carries a rationale string, so
#        a human reviewer can audit *why* — not just *what*.
#
# All example content and code are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Re-establish foundation from Notebook 00 (self-contained bootstrap)
#
# Notebooks cannot share live Python objects, so we rebuild the minimal set of
# paths, API client, schema, and llm_call() here. This mirrors 00 exactly and
# lets Notebook 01 run standalone.
# =============================================================================
import os
import re
import json
import time
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# --- Paths -------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW  = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"
INFER     = ARTIFACTS / "inference"
TAB       = ARTIFACTS / "tables"
CACHE     = ARTIFACTS / "llm_cache"
for p in (INFER, TAB, CACHE):
    p.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


SEED = 42
np.random.seed(SEED)

# --- Load run manifest written by Notebook 00 --------------------------------
RUN_MANIFEST = ARTIFACTS / "run_manifest.json"
if not RUN_MANIFEST.exists():
    raise FileNotFoundError(
        "[ERROR] artifacts/run_manifest.json not found. "
        "Run 00_setup_and_data.ipynb first."
    )
manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))

MODELS       = manifest["models"]
DEFAULT_TIER = manifest["default_tier"]
AUTO_GRADES  = manifest["auto_grades"]
MISSING      = manifest["missing_sentinel"]

print(f"[INFO] Manifest loaded. default_tier={DEFAULT_TIER}, "
      f"grades={list(AUTO_GRADES)}")

# --- API client --------------------------------------------------------------
load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env at project root.")
client = OpenAI(api_key=OPENAI_API_KEY)


# %%
# =============================================================================
# Cell 2. Re-declare schema, cost tracker, and llm_call() (identical to nb 00)
# =============================================================================
@dataclass
class WorkUnit:
    """One extracted task. Domain-neutral fields only."""
    id: str
    name: str
    actor: str = MISSING
    system: str = MISSING
    description: str = ""
    auto_grade: str = MISSING
    auto_rationale: str = ""
    minutes_per_case: Any = MISSING
    cases_per_month: Any = MISSING
    monthly_minutes: Any = MISSING
    time_rationale: str = ""
    evidence: str = ""

    def to_dict(self) -> dict:
        return asdict(self)


class CostTracker:
    def __init__(self):
        self.records: list[dict] = []

    def add(self, tier, model, usage, tag=""):
        p = MODELS.get(tier, {})
        pin, pcached, pout = p.get("in"), p.get("cached_in"), p.get("out")
        pt = getattr(usage, "prompt_tokens", 0) or 0
        ct = getattr(usage, "completion_tokens", 0) or 0
        cached = 0
        det = getattr(usage, "prompt_tokens_details", None)
        if det is not None:
            cached = getattr(det, "cached_tokens", 0) or 0
        fresh = max(pt - cached, 0)
        cost = None
        if None not in (pin, pout):
            pc = pcached if pcached is not None else pin
            cost = (fresh * pin + cached * pc + ct * pout) / 1_000_000
        self.records.append({
            "tag": tag, "tier": tier, "model": model,
            "prompt_tokens": pt, "cached_tokens": cached,
            "completion_tokens": ct, "cost_usd": cost,
        })
        return cost or 0.0

    def summary(self):
        return pd.DataFrame(self.records) if self.records else pd.DataFrame()

    def total_usd(self):
        return float(sum(r["cost_usd"] or 0.0 for r in self.records))

    def flush(self, stage: str):
        """Append this notebook's spend to a shared ledger so the five-axis
        Efficiency computation and the dashboard can read the TRUE cumulative
        pipeline cost across all notebooks (they do not share memory)."""
        ledger = ARTIFACTS / "cost_ledger.json"
        data = json.loads(ledger.read_text(encoding="utf-8")) if ledger.exists() else {}
        data[stage] = {
            "total_usd": round(self.total_usd(), 6),
            "n_calls": len(self.records),
        }
        ledger.write_text(json.dumps(data, ensure_ascii=False, indent=2),
                          encoding="utf-8")
        return data


COST = CostTracker()


def _safe_json(text):
    if not text:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    for opener, closer in (("{", "}"), ("[", "]")):
        i, j = t.find(opener), t.rfind(closer)
        if 0 <= i < j:
            try:
                return json.loads(t[i:j + 1])
            except Exception:
                continue
    return None


def _cache_key(model, system, user, temperature, response_json):
    raw = json.dumps({"m": model, "s": system, "u": user,
                      "t": temperature, "j": response_json},
                     ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def llm_call(system, user, tier=DEFAULT_TIER, temperature=0.0,
             response_json=True, tag="", use_cache=True, max_retries=3):
    model = MODELS[tier]["name"]
    key = _cache_key(model, system, user, temperature, response_json)
    cache_file = CACHE / f"{key}.json"
    if use_cache and cache_file.exists():
        c = json.loads(cache_file.read_text(encoding="utf-8"))
        c["cached"] = True
        c["cost_usd"] = 0.0
        return c
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
    }
    if response_json:
        kwargs["response_format"] = {"type": "json_object"}
    kwargs["temperature"] = temperature
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            parsed = _safe_json(text) if response_json else None
            if response_json and parsed is None:
                raise ValueError("Response was not valid JSON.")
            cost = COST.add(tier, model, resp.usage, tag=tag or model)
            out = {"text": text, "json": parsed, "cached": False,
                   "tier": tier, "model": model, "cost_usd": cost}
            if use_cache:
                cache_file.write_text(json.dumps(out, ensure_ascii=False),
                                      encoding="utf-8")
            return out
        except TypeError as e:
            if "temperature" in kwargs:
                kwargs.pop("temperature", None)
                last_err = e
                continue
            last_err = e
        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"[llm_call] failed after {max_retries} retries: {last_err}")


print("[INFO] Schema, CostTracker, llm_call() re-established for nb 01.")


# %%
# =============================================================================
# Cell 3. Load interview text (the sole domain input to Agent 1)
# =============================================================================
INTERVIEW_PATH = DATA_RAW / manifest["paths"]["interview"].split("/")[-1] \
    if "/" in manifest["paths"]["interview"] else DATA_RAW / "interview_tcb.txt"
# Robust fallback to the known filename:
if not INTERVIEW_PATH.exists():
    INTERVIEW_PATH = DATA_RAW / "interview_tcb.txt"

INTERVIEW_TEXT = INTERVIEW_PATH.read_text(encoding="utf-8").strip()
print(f"[INFO] Interview: {len(INTERVIEW_TEXT)} chars from {rel(INTERVIEW_PATH)}")


# %%
# =============================================================================
# Cell 4. Agent-1 prompt (domain-agnostic; Chain-of-Thought + strict JSON)
#
# The prompt describes a GENERIC task — "extract repeatable work units from any
# operational interview and grade their AI-automatability" — with no reference
# to the specific domain. Swapping the interview swaps the domain; the prompt is
# unchanged. This is what makes the artifact transferable.
# =============================================================================
AGENT1_SYSTEM = f"""\
You are Agent 1 (Task Extraction & Automatability Classification) in a pipeline
that estimates the ROI of adopting generative AI for knowledge work. You analyze
an operational interview transcript from ANY business domain and identify the
discrete, repeatable work units it describes.

Reason step by step (chain-of-thought) BEFORE producing the final answer:
  1. Read the transcript and list every distinct task an actor performs.
  2. Merge duplicates; split compound tasks into atomic work units.
  3. For each unit, decide who performs it and what system/tool is used, IF and
     ONLY IF the transcript states it. If not stated, use the sentinel "{MISSING}".
  4. Grade each unit's AI-automatability using exactly one of:
       - "full"    : {AUTO_GRADES['full']}
       - "partial" : {AUTO_GRADES['partial']}
       - "manual"  : {AUTO_GRADES['manual']}
  5. Write a one-sentence rationale justifying the grade, grounded in the task's
     nature (data-driven vs. judgment vs. physical), not in domain assumptions.

Rules:
  - Do NOT invent tasks, actors, systems, times, or volumes not in the text.
  - Do NOT estimate durations or costs — that is a later agent's job.
  - Physical, in-person, or non-digitizable steps must be graded "manual".
  - Steps that only require reading text and producing structured/textual output
    are typically "full"; steps needing human judgment or sign-off are "partial".

Return ONLY a JSON object with this exact shape:
{{
  "reasoning": "<your concise step-by-step reasoning>",
  "work_units": [
    {{
      "id": "w1",
      "name": "<short task label>",
      "actor": "<role/team or {MISSING}>",
      "system": "<tool/system or {MISSING}>",
      "description": "<one-line paraphrase of the task>",
      "auto_grade": "full | partial | manual",
      "auto_rationale": "<one sentence: why this grade>",
      "evidence": "<short phrase from the transcript this unit came from>"
    }}
  ]
}}
"""

AGENT1_USER = f"""\
Here is the operational interview transcript. Extract and classify all work units.

--- BEGIN TRANSCRIPT ---
{INTERVIEW_TEXT}
--- END TRANSCRIPT ---
"""

print(f"[INFO] Agent-1 prompt ready "
      f"(system {len(AGENT1_SYSTEM)} chars, user {len(AGENT1_USER)} chars).")


# %%
# =============================================================================
# Cell 5. Run Agent 1 (deterministic: temperature = 0.0)
# =============================================================================
CFG = manifest["pipeline_cfg"]["agent1_extract"]  # {tier, temperature, n_samples}

result = llm_call(
    system=AGENT1_SYSTEM,
    user=AGENT1_USER,
    tier=CFG["tier"],
    temperature=CFG["temperature"],
    response_json=True,
    tag="agent1_extract",
)

payload = result["json"] or {}
raw_units = payload.get("work_units", [])
print(f"[INFO] Agent 1 returned {len(raw_units)} work units "
      f"(cached={result['cached']}, cost=${result['cost_usd']:.5f}).")
print(f"[INFO] Model reasoning (preview): "
      f"{str(payload.get('reasoning',''))[:300]}...")


# %%
# =============================================================================
# Cell 6. Normalize into the WorkUnit schema + validate (quality gate DP1)
#
# A validation gate between agents localizes errors: we coerce every record to
# the WorkUnit schema, enforce valid grades, assign stable ids, and flag any
# unit the model returned in an unexpected shape.
# =============================================================================
def normalize_units(raw: list[dict]) -> tuple[list[WorkUnit], list[str]]:
    units: list[WorkUnit] = []
    problems: list[str] = []
    valid_grades = set(AUTO_GRADES)

    for i, r in enumerate(raw, start=1):
        if not isinstance(r, dict) or not r.get("name"):
            problems.append(f"record {i}: missing/invalid 'name' -> skipped")
            continue
        grade = str(r.get("auto_grade", MISSING)).strip().lower()
        if grade not in valid_grades:
            problems.append(
                f"record {i} ('{r.get('name')}'): invalid grade "
                f"'{r.get('auto_grade')}' -> set to {MISSING}"
            )
            grade = MISSING
        units.append(WorkUnit(
            id=r.get("id") or f"w{i}",
            name=str(r.get("name")).strip(),
            actor=str(r.get("actor", MISSING)).strip() or MISSING,
            system=str(r.get("system", MISSING)).strip() or MISSING,
            description=str(r.get("description", "")).strip(),
            auto_grade=grade,
            auto_rationale=str(r.get("auto_rationale", "")).strip(),
            evidence=str(r.get("evidence", "")).strip(),
        ))
    # Re-assign stable sequential ids so downstream agents can rely on them.
    for k, u in enumerate(units, start=1):
        u.id = f"w{k}"
    return units, problems


work_units, problems = normalize_units(raw_units)

print(f"[INFO] Normalized {len(work_units)} work units.")
if problems:
    print(f"[WARN] {len(problems)} validation issue(s):")
    for p in problems:
        print("   -", p)
else:
    print("[INFO] All records passed the validation gate.")


# %%
# =============================================================================
# Cell 7. Inspect results as a table (paper-ready view)
# =============================================================================
df1 = pd.DataFrame([u.to_dict() for u in work_units])
view_cols = ["id", "name", "actor", "system", "auto_grade", "auto_rationale"]
df1_view = df1[view_cols] if not df1.empty else df1

with pd.option_context("display.max_colwidth", 60, "display.width", 160):
    print(df1_view.to_string(index=False))

# Grade distribution — a first sanity check on the extraction.
if not df1.empty:
    dist = df1["auto_grade"].value_counts().reindex(
        list(AUTO_GRADES) + [MISSING]).fillna(0).astype(int)
    print("\n[INFO] Automatability grade distribution:")
    for g, n in dist.items():
        print(f"   {g:8s}: {n}")


# %%
# =============================================================================
# Cell 8. Persist Agent-1 output for the next stage (Agent 2) + cost log
# =============================================================================
OUT_PATH = INFER / "agent1_work_units.json"
out_obj = {
    "agent": "agent1_task_extraction",
    "interview_source": rel(INTERVIEW_PATH),
    "model": result["model"],
    "tier": CFG["tier"],
    "temperature": CFG["temperature"],
    "n_work_units": len(work_units),
    "reasoning": payload.get("reasoning", ""),
    "work_units": [u.to_dict() for u in work_units],
    "validation_problems": problems,
}
OUT_PATH.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2),
                    encoding="utf-8")
print(f"[INFO] Agent-1 output -> {rel(OUT_PATH)}  ({len(work_units)} units)")

# Also export a CSV table for the paper's appendix.
if not df1.empty:
    csv_path = TAB / "agent1_work_units.csv"
    df1[view_cols + ["evidence", "description"]].to_csv(
        csv_path, index=False, encoding="utf-8-sig")
    print(f"[INFO] Paper table -> {rel(csv_path)}")

print(f"[INFO] Agent-1 spend this run: ${COST.total_usd():.5f}")
COST.flush("01_agent1")   # append to shared cost ledger

[INFO] Manifest loaded. default_tier=weak, grades=['full', 'partial', 'manual']
[INFO] Schema, CostTracker, llm_call() re-established for nb 01.
[INFO] Interview: 5062 chars from data\raw\interview_tcb.txt
[INFO] Agent-1 prompt ready (system 2042 chars, user 5195 chars).
[INFO] Agent 1 returned 46 work units (cached=True, cost=$0.00000).
[INFO] Model reasoning (preview): I analyzed the transcript to identify distinct tasks performed by various actors, ensuring to merge duplicates and split compound tasks into atomic units. Each task was then classified based on the information provided regarding who performs it and what systems are used, while also assessing the lev...
[INFO] Normalized 46 work units.
[INFO] All records passed the validation gate.
 id                                                          name                   actor                               system auto_grade                                                                        auto_rationale
 w1               

{'01_agent1': {'total_usd': 0.0, 'n_calls': 0}}